# HW05 Data Science Academy
# Мурзабеков Эльдар 

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.

In [92]:
import pandas as pd

Ваши ответы и объяснения:

1. Будет создан series с индексами a, b, c и значениями 10, 20, 30. print выведет 20 из индекса b и первое значение 10 с помощью iloc[0]

In [93]:
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])

20 10


2. Объединит series таблицы по индексам. Шымкент есть в двух таблицах, поэтому значение для него суммируется. Для Астаны и Алматы будет NaN. Вывод будет: Астана : Nan, Алматы : Nan, Шымкент : 3

In [94]:
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)

Алматы     NaN
Астана     NaN
Шымкент    3.0
dtype: float64


3. С помощью маски фильтруем датафрейм по x, где значения больше 1 и получаем shape. Вывод будет: 2 2 для x и y

In [95]:
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)

(2, 2)


4. В первом print вывод будет series, во втором dataframe, так-как используется на квадратную скобку больше для запроса таблицы

In [96]:
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)

Series DataFrame


5. С помощью маски фильтруем x, где значения больше 1. Затем выводим в виде листа по оставшимся отфильтрованным позициям. Вывод будет: индекс [1, 2] и для iloc позция 5 

In [97]:
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])

[1, 2] 5


In [98]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [99]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [100]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [101]:
print(f"Количество строк: {df.shape[0]}") 
print(f"Количество столбцов: {df.shape[1]}")
print("Типы столбцов:"); df.info()
print(f"Пропуски в delivery_days: {np.sum(df['delivery_days'].isna())}")
print(f"Количество разных значений в city: {df['city'].nunique()}")
print(f"Количество разных значений в category: {df['category'].nunique()}")
print(f"Количество разных значений в status: {df['status'].nunique()}")
print(f"Количество разных значений в customer_id: {df['customer_id'].nunique()}")

Количество строк: 5463
Количество столбцов: 8
Типы столбцов:
<class 'pandas.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   str    
 2   customer_id    5463 non-null   str    
 3   city           5463 non-null   str    
 4   category       5463 non-null   str    
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 341.6 KB
Пропуски в delivery_days: 160
Количество разных значений в city: 14
Количество разных значений в category: 5
Количество разных значений в status: 2
Количество разных значений в customer_id: 2003


Список проблем:

1) Название одного города может быть написано несколько раз по разному, например: 'Астана', 'астана', 'Astana ' , что искажает данные, нужно нормализовать и привести к единому виду

2) В данных также присутствуют пользователи TEST, которые могут привести к выбросам данных и искажать средний чек/выручку, нужно исключить их из метрик

3) Есть повторы в order_id которые могут завысить заказы и выручку, нужно оставлять только уникальные заказы без дубликатов для метрик

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [102]:
print(f"Среднее для оплаченных заказов без TEST: {df['amount'][(paid) & (not_test)].mean():.2f}")
print(f"Медиана для оплаченных заказов без TEST: {df['amount'][(paid) & (not_test)].median():.2f}")
print(f"75-й перцентиль без TEST: {np.percentile(df['amount'][(paid) & (not_test)], 75)}")
print(f"25-й перцентиль без TEST: {np.percentile(df['amount'][(paid) & (not_test)], 25)}")

Среднее для оплаченных заказов без TEST: 14672.39
Медиана для оплаченных заказов без TEST: 10300.00
75-й перцентиль без TEST: 18690.0
25-й перцентиль без TEST: 5880.0


Типичным чеком является медиана со значением 10300, так-как большие значения могут исказить среднее. Медиана более честная для отчета, ибо она менее чувствительна к выбросам

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [103]:
ATA = ['Алматы', 'алматы', 'АЛМАТЫ',' Алматы', 'Almaty']
sh_total = df['city'].isin(SH).sum()
sh_cancelled = ((df['city'].isin(SH)) & (df['status'] == 'отменён')).sum()
ata_total = df['city'].isin(ATA).sum()
ata_cancelled = ((df['city'].isin(ATA)) & (df['status'] == 'отменён')).sum()
print(f"Доля отмененных заказов по всем городам: {df['city'][df['status'] == 'отменён'].count() / df.shape[0] * 100:.2f}%") 
print(f"Доля отмененных заказов в Шымкенте: {sh_cancelled / sh_total * 100:.2f}%")
print(f"Доля отмененных заказов в Алматы: {ata_cancelled / ata_total * 100:.2f}%")

Доля отмененных заказов по всем городам: 10.23%
Доля отмененных заказов в Шымкенте: 20.40%
Доля отмененных заказов в Алматы: 7.86%


Среди всех городов выделяется Шымкент, здесь доля отменённых заказов достигает 20.40, что почти в 2 раза выше среднего показателя по всем городам 10.23 и в 2.6 раза выше, чем в Алмате 7.86

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [104]:
sh_sep = df['amount'][(df['city'].isin(SH)) & paid & sep & not_test].sum()
sh_oct = df['amount'][(df['city'].isin(SH)) & paid & octo & not_test].sum()
ata_sep = df['amount'][(df['city'].isin(ATA)) & paid & sep & not_test].sum()
ata_oct = df['amount'][(df['city'].isin(ATA)) & paid & octo & not_test].sum()
print(f"Выручка Шымкента за сентябрь: {sh_sep}")
print(f"Выручка Шымкента за октябрь: {sh_oct}")
print(f"Процентное изменение в Шымкенте: {(sh_oct - sh_sep) / sh_sep * 100:.2f}%")
print(f"Выручка Алматы за сентябрь: {ata_sep}")
print(f"Выручка Алматы за октябрь: {ata_oct}")
print(f"Процентное изменение в Алмате: {(ata_oct - ata_sep) / ata_sep * 100:.2f}%")

Выручка Шымкента за сентябрь: 4949800.0
Выручка Шымкента за октябрь: 4044010.0
Процентное изменение в Шымкенте: -18.30%
Выручка Алматы за сентябрь: 15523940.0
Выручка Алматы за октябрь: 14643020.0
Процентное изменение в Алмате: -5.67%


Выручка в Шымкенте снизилась на 18.30%, в Алмате 5.67%. Падение выручки в Шымкенте примерно в 3.23 раза сильнее, чем в Алматы

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [105]:
cancel_sh = ((df['city'].isin(SH)) & (df['status'] == 'отменён')).sum()
print(f"Отмененные заказы в Шымкенте: {cancel_sh}")

Отмененные заказы в Шымкенте: 164


Две ошибки:

1) Ошибка в коде: отдельные условия не заключены в скобки. Из-за этого Python неверно определяет порядок их выполнения, ибо оператор & имеет более высокий приоритет чем ==

2) Логическая ошибка: проверка учитывает только вариант «Шымкент», поэтому другие написания, например «шымкент» и «Shymkent», остаются без внимания. Для корректной проверки лучше использовать .isin(SH)

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [106]:
most_orders = df.value_counts('customer_id').index[0]
print(f"Больше всего заказов у клиента: {most_orders}")
print(df.loc[df['customer_id'] == most_orders][['order_id', 'order_date', 'city', 'amount', 'status']])
print(f"Количество заказов: {(df['customer_id'] == most_orders).sum()}")
print(f"Количество заказов без дубликатов: {df.loc[df['customer_id'] == most_orders]['order_id'].nunique()}")

Больше всего заказов у клиента: C01818
      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен
Количество заказов: 11
Количество заказов без дубликатов: 10


В таблице дважды указан заказ с номером 101504. Сумма каждой записи составляет 8890 тг. Если это один и тот же заказ, то в расчёте выручки сумма учитывается два раза. В результате общая выручка завышается на 8890 тг. Поэтому стоит проверить, нет ли таких же повторов в остальных данных.

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [107]:
Q1 = np.percentile(df['amount'][(not_test) & (paid)], 25)
Q3 = np.percentile(df['amount'][(not_test) & (paid)], 75)
IQR = Q3 - Q1
border = Q3 + 1.5 * IQR
bigger = np.sum(df['amount'][(not_test) & (paid)] > border)
mostcat = df['category'][(df['amount'] > border) & (not_test) & (paid)]
print(f"IQR: {IQR}")
print(f"Граница: {border}")
print(f"Количество заказов выше границы: {bigger}")
print(f"Доля выбросов: {bigger / (not_test & paid).sum() * 100:.2f}%")
print(f"Распределение выбросов по категориям: {mostcat.value_counts()}")

IQR: 12810.0
Граница: 37905.0
Количество заказов выше границы: 320
Доля выбросов: 6.53%
Распределение выбросов по категориям: category
Электроника    308
Одежда          10
Дом              2
Name: count, dtype: int64


Метод IQR выявил 320 выбросов (6,53%), а Z-score обнаружил только 87 (1,84%) из дз4. Такая разница связана с особенностями методов. В наших данных сумма заказа имеет правостороннюю асимметрию, поэтому крупные заказы повышают среднее значение и стандартное отклонение. Из-за этого Z-score может не учитывать некоторые высокие значения как выбросы. IQR использует квартили, поэтому определяет больше выбросов. 

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [108]:
total_cancelled = df['city'][df['status'] == 'отменён'].value_counts()
total_orders = df['city'].value_counts()
print(f"Отмененные заказы: {total_cancelled}")
print(f"Все заказы вне зависимости от статуса: {total_orders}")
print(f"Доля отмен: {(total_cancelled / total_orders).sort_values()}") 

Отмененные заказы: city
Алматы       154
Шымкент      153
Астана       104
Караганда     48
Актобе        41
Атырау        11
астана        11
алматы         9
Shymkent       9
Astana         8
Almaty         4
 Алматы        3
АЛМАТЫ         2
шымкент        2
Name: count, dtype: int64
Все заказы вне зависимости от статуса: city
Алматы       1927
Астана       1226
Шымкент       711
Караганда     538
Актобе        397
Атырау        146
астана         84
Astana         80
Almaty         69
 Алматы        66
алматы         64
АЛМАТЫ         62
шымкент        54
Shymkent       39
Name: count, dtype: int64
Доля отмен: city
АЛМАТЫ       0.032258
шымкент      0.037037
 Алматы      0.045455
Almaty       0.057971
Атырау       0.075342
Алматы       0.079917
Астана       0.084829
Караганда    0.089219
Astana       0.100000
Актобе       0.103275
астана       0.130952
алматы       0.140625
Шымкент      0.215190
Shymkent     0.230769
Name: count, dtype: float64


Шымкент: 711 заказов, шымкент: 54 заказа, Shymkent: 39 заказов.

1) Судить доли по одному названию нельзя, ибо есть Шымкент, шымкент и Shymkent с разными значениями 

2) Нужно нормализовать названия городов перед анализом, привести к единому виду. Убрать лишние пробелы, привести регистр к общему виду, объединить русские и английские названия городов к одному названию 